In [1]:
import numpy as np
from DCC import DCC

example_input = '4pfs;B;ANP ADP;19 20 21 22 23 24 25 26 48 135 192 220 221 222 223'
pdb_id, chain_id, binding_ligands_str, binding_residues_str = example_input.split(';')
binding_ligands = binding_ligands_str.split()
binding_residues = [int(x) for x in binding_residues_str.split()]
predicted_residues = [191, 192, 217, 218, 219, 220, 221, 222, 223]

path = '/home/vit/Projects/deeplife-2026/data/4pfs.cif'

def get_coordinates(cif_file_path, chain_id, binding_residues, auth=True):
    import biotite.structure.io.pdbx as pdbx
    from biotite.structure.io.pdbx import get_structure
    from biotite.structure import get_residues

    cif_file = pdbx.CIFFile.read(cif_file_path)
    
    protein = get_structure(cif_file, model=1, use_author_fields=auth)    
    protein = protein[(protein.atom_name == "CA") 
                        & (protein.element == "C") 
                        & (protein.chain_id == chain_id) ]

    residue_ids, residue_types = get_residues(protein)
    coordinates = []
    for i in range(len(residue_ids)):
        if residue_ids[i] in binding_residues:
            coordinates.append(protein[i].coord)
    return np.array(coordinates)

def compute_center(points):
    return np.mean(points, axis=0)

## DCC
Demonstration of DCC.

### Guideline
Compute DCC for every actual pocket in the dataset, and estimate how many DCC values are less then 12Å. Report the % of successfully predicted pockets.

In [2]:
actual_center_of_pocket = compute_center(get_coordinates(path, chain_id, binding_residues))
predicted_center_of_pocket = compute_center(get_coordinates(path, chain_id, predicted_residues))

print(f'The actual center of the pocket is: {actual_center_of_pocket}')
print(f'The distance between the predicted and actual centers is: {DCC([predicted_center_of_pocket], [actual_center_of_pocket])}')
print()

print('Adding artificial data for demonstration ...')
# you can run DCC(...) for all predictions at once, let's say you have multiple predicted pockets and multiple actual pockets:
predicted_pockets = [predicted_center_of_pocket, np.array([0, 0, 0]), np.array([1, 1, 1])] # add artificial data for demonstration: np.array([0, 0, 0]), np.array([1, 1, 1])
actual_pockets = [actual_center_of_pocket, np.array([0, 0, 15])] # add artificial data for demonstration: np.array([0, 0, 10])
dcc_values = DCC(predicted_pockets, actual_pockets)
print(f'DCC values for all predicted pockets: {dcc_values}')
print(f"Let's say your 'success' threshold is 12 Å, then the first pocket is a success, while the second is a failure (>12 Å) -> DCC = 50%")

The actual center of the pocket is: [ -5.118067   9.167133 -42.038067]
The distance between the predicted and actual centers is: [9.800156]

Adding artificial data for demonstration ...
DCC values for all predicted pockets: [9.800156, 14.071247279470288]
Let's say your 'success' threshold is 12 Å, then the first pocket is a success, while the second is a failure (>12 Å) -> DCC = 50%


# RRO
Relative residue overlap is determined by number of residues that are covering the pocket.

In [4]:
from RRO import RRO
rro_values = RRO([predicted_residues], [binding_residues])
print(f'RRO values for the predicted pocket: {rro_values}')

RRO values for the predicted pocket: [0.3333333333333333]
